In [21]:
import pandas as pd
from pathlib import Path
import numpy as np
import glob
from os.path import join, basename
from geopy.distance import geodesic
import unicodedata
import os
import zipfile


In [22]:
def normalizar_nome(nome):
    if pd.isna(nome):
        return ""
    nome_limpo = str(nome).strip().lower()
    nome_limpo = nome_limpo.replace(' (wmo)', '').replace(' ', '_').replace('-', ' ').replace(':', '')
    nome_limpo = ''.join(
        c for c in unicodedata.normalize('NFD', nome_limpo)
        if unicodedata.category(c) != 'Mn'
    )
    return " ".join(nome_limpo.split())

In [23]:
def read_stations_info(input_folder, stations_codes):
    zip_files = glob.glob(os.path.join(input_folder, "*.zip"))

    stations_info = {}

    for zip_file in zip_files:
        file_name = os.path.basename(zip_file)
        if int(file_name.replace('.zip', '')) >= 2003 and int(file_name.replace('.zip', '')) <= 2024:
            with zipfile.ZipFile(zip_file, 'r') as z:
                csv_names = [n for n in z.namelist() if n.endswith('.CSV') and any(code in n for code in stations_codes)]
                for csv_name in csv_names:
                    with z.open(csv_name) as csv_file:
                        df = pd.read_csv(csv_file, nrows=7, encoding='ISO-8859-1', sep=';', header=None).T
                        df.columns = [normalizar_nome(col) for col in df.iloc[0]]
                        df = df[1:].iloc[0].to_dict()
                        stations_info.update({df['codigo']: df})
    return stations_info

In [24]:
def extracts_stations_info(input_folder, stations_codes):
    zip_files = glob.glob(os.path.join(input_folder, "*.zip"))

    root_folder = 'inmet'

    for station_code in stations_codes:
        os.makedirs(join(root_folder, 'extracted', station_code), exist_ok=True)
        for zip_file in zip_files:
            file_name = basename(zip_file)
            file_year = int(file_name.replace('.zip', ''))
            if file_year >= 2003 and file_year <= 2024:
                with zipfile.ZipFile(zip_file, 'r') as z:
                    csv_names = [n for n in z.namelist() if n.endswith('.CSV') and station_code in n]
                    for csv_name in csv_names:
                        print(f"Extracting {csv_name} to {join(root_folder, station_code)}")
                        z.extract(csv_name, join(root_folder, 'extracted', station_code))    

In [25]:
def read_inmet_data(input_folder):
    columns_name = [
        "date",
        "utc_hour",
        "precipitation_mm",
        "station_pressure_mb",
        "max_pressure_mb",
        "min_pressure_mb",
        "global_radiation_kj_m2",
        "air_temperature_c",
        "dew_point_temperature_c",
        "max_temperature_c",
        "min_temperature_c",
        "max_dew_point_c",
        "min_dew_point_c",
        "max_relative_humidity_pct",
        "min_relative_humidity_pct",
        "relative_humidity_pct",
        "wind_direction_degrees",
        "max_wind_gust_ms",
        "wind_speed_ms",
        "temp_col"
    ]

    path = Path(input_folder) 

    stations_data = {}

    # List only directories
    folders = [f.name for f in path.iterdir() if f.is_dir()]
    for station in folders:

        full_timeline = pd.date_range(start='2003-01-01', end='2024-12-31', freq='h')

        search_dir = Path(input_folder) / station
        csv_files = [str(f) for f in search_dir.rglob("*.CSV")]
        
        print(f"Processing station {station} with {len(csv_files)} files.")
        df = pd.concat((pd.read_csv(csv, sep=';', skiprows=9, encoding='ISO-8859-1', header=None, names=columns_name, on_bad_lines='warn') for csv in csv_files), ignore_index=True).drop(columns=['temp_col'])
            
        df['date'] = df['date'].str.replace('/', '-', regex=False)
        mask = df['utc_hour'].str.contains('UTC', na=False)
        df.loc[mask, 'utc_hour'] = pd.to_datetime(df.loc[mask, 'utc_hour'], format='%H%M UTC').dt.strftime('%H:%M')
        
        df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['utc_hour'], format='%Y-%m-%d %H:%M')
        
        cols = df.columns.drop(['datetime', 'date', 'utc_hour'])

        df[cols] = df[cols].apply(lambda x: pd.to_numeric(x.astype(str).str.replace(',', '.'), errors='coerce'))
        
        df = df.replace(-9999.0, np.nan)
        
        # Corrects the radiation along the night
        condicao = ((df['datetime'].dt.hour >= 19) | (df['datetime'].dt.hour <= 5)) & (df['global_radiation_kj_m2'].isnull())
        df.loc[condicao, 'global_radiation_kj_m2'] = 0
        
        df = df[['precipitation_mm', 'air_temperature_c', 'relative_humidity_pct', 'dew_point_temperature_c', 'global_radiation_kj_m2', 'datetime']]
        
        df.set_index('datetime', inplace=True)

        df = df.reindex(full_timeline)
        
        stations_data[station] = df
    
    return stations_data


In [26]:
def calculate_closest_stations(stations_info):

    for station, info in stations_info.items():
        coords = (float(info['latitude'].replace(',', '.')), float(info['longitude'].replace(',', '.')))

        sorted_neighbors = sorted(
            [k for k in stations_info if k != station],
            key=lambda k: geodesic(coords, (float(stations_info[k]['latitude'].replace(',', '.')), float(stations_info[k]['longitude'].replace(',', '.')))).kilometers
        )

        stations_info[station]['most_closest'] = sorted_neighbors

    
    return stations_info
    
    

In [27]:
def fill_imnet_dataset(stations_info, stations_data):
    inmet_filled = {}

    for station, info in stations_info.items():
        df_atual = stations_data[station].copy()
        
        print(f"Processando estação: {station}")
        
        for neighbor in info['most_closest']:
            print(f"  -> Filling station {station} with neighbor {neighbor}")
            df_atual = df_atual.combine_first(stations_data[neighbor])
                
        inmet_filled[station] = df_atual
    
    return inmet_filled

In [40]:
def process_inmet_data(df_raw, ibge_code, city):

    df_filtered = df_raw[(df_raw.index.month >= 1) & (df_raw.index.month <= 7)].copy()

    df_filtered['year'] = df_filtered.index.year

    'precipitation_mm', 'air_temperature_c', 'relative_humidity_pct', 'dew_point_temperature_c', 'global_radiation_kj_m2', 'datetime'

    yearly_summary = df_filtered.groupby(['year']).agg(
        mean_temperature_c=('air_temperature_c', 'mean'),  
        max_temperature_c=('air_temperature_c', 'max'),    
        min_temperature_c=('air_temperature_c', 'min'),    
        total_rain_mm=('precipitation_mm', 'sum'),
        sum_global_radiation_kj_m2=('global_radiation_kj_m2', 'sum'),
        mean_relative_humidity_pct=('relative_humidity_pct', 'mean'),
        max_relative_humidity_pct=('relative_humidity_pct', 'max'),
        min_relative_humidity_pct=('relative_humidity_pct', 'min')
    ).reset_index().round(2)

    yearly_summary['code'] = int(ibge_code)
    yearly_summary['city'] = city
    
    return yearly_summary

In [29]:
def assign_station_to_citie(inmet_filled, stations_info, cities_coords):
    city_inmet_data = {}
    info = {}
    for coord in cities_coords.itertuples():
        ibge_coord = (coord.latitude, coord.longitude)
        closest_station = min(
            stations_info.keys(),
            key=lambda k: geodesic(ibge_coord, (float(stations_info[k]['latitude'].replace(',', '.')), float(stations_info[k]['longitude'].replace(',', '.')))).kilometers
        )
        info[coord.code] = {
            'city_name': coord.name,
            'closest_station': closest_station,
            'distance_km': geodesic(ibge_coord, (float(stations_info[closest_station]['latitude'].replace(',', '.')), float(stations_info[closest_station]['longitude'].replace(',', '.')))).kilometers
        }
        city_inmet_data[coord.code] = process_inmet_data(inmet_filled[closest_station], coord.code, coord.name)
    
    return city_inmet_data

### Filtering the inmet stations for the desired period and locality

In [30]:
inmet_estacoes = pd.read_csv('inmet/inmet_estacoes.csv', sep='\t')
inmet_estacoes['data_instalacao'] = pd.to_datetime(inmet_estacoes['Data de Instalação'], errors='coerce')
stations_codes = inmet_estacoes[(inmet_estacoes['UF'] == 'MT') & (inmet_estacoes['data_instalacao'].dt.year >= 2000) & (inmet_estacoes['data_instalacao'].dt.year <= 2024)]['Código'].tolist()

C:\Users\franc\AppData\Local\Temp\ipykernel_45432\3719105132.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  inmet_estacoes['data_instalacao'] = pd.to_datetime(inmet_estacoes['Data de Instalação'], errors='coerce')


### Reading the informations from the cities analized

In [31]:
cities_coords = pd.read_csv('ibge/cities_to_analize.csv')

### Reading the information for the select stations directly from the csv with data

In [32]:
stations_info = read_stations_info('../Data Preparation/Dados', stations_codes)

### Extracting the information to a more readble format

In [33]:
# extracts_stations_info('../Data Preparation/Dados', stations_codes)

### Reading the data for each station

In [34]:
stations_data = read_inmet_data('inmet/extracted')

Processing station A901 with 22 files.
Processing station A902 with 22 files.
Processing station A903 with 22 files.
Processing station A904 with 22 files.
Processing station A905 with 22 files.
Processing station A906 with 22 files.
Processing station A907 with 22 files.
Processing station A908 with 19 files.
Processing station A909 with 14 files.
Processing station A910 with 19 files.
Processing station A911 with 8 files.
Processing station A912 with 19 files.
Processing station A914 with 19 files.
Processing station A916 with 16 files.
Processing station A917 with 19 files.
Processing station A919 with 18 files.
Processing station A920 with 19 files.
Processing station A922 with 19 files.
Processing station A923 with 8 files.
Processing station A924 with 18 files.
Processing station A926 with 17 files.
Processing station A927 with 17 files.
Processing station A928 with 17 files.
Processing station A929 with 17 files.
Processing station A930 with 17 files.
Processing station A931 wit

### Calculating the distance between the stations

In [35]:
stations_info = calculate_closest_stations(stations_info)

### Filling the empty years and the null values with the closest station data

In [36]:
inmet_filled = fill_imnet_dataset(stations_info, stations_data)

Processando estação: A901
  -> Filling station A901 with neighbor A912
  -> Filling station A901 with neighbor A944
  -> Filling station A901 with neighbor A935
  -> Filling station A901 with neighbor A923
  -> Filling station A901 with neighbor A902
  -> Filling station A901 with neighbor A941
  -> Filling station A901 with neighbor A907
  -> Filling station A901 with neighbor A936
  -> Filling station A901 with neighbor A933
  -> Filling station A901 with neighbor A931
  -> Filling station A901 with neighbor A903
  -> Filling station A901 with neighbor A932
  -> Filling station A901 with neighbor A905
  -> Filling station A901 with neighbor A929
  -> Filling station A901 with neighbor A928
  -> Filling station A901 with neighbor A904
  -> Filling station A901 with neighbor A937
  -> Filling station A901 with neighbor A909
  -> Filling station A901 with neighbor A934
  -> Filling station A901 with neighbor A911
  -> Filling station A901 with neighbor A930
  -> Filling station A901 wit

### Assign a station data to a city

In [41]:
imnet_by_citie = assign_station_to_citie(inmet_filled, stations_info, cities_coords)

In [ ]:
df_inmet_final = pd.concat(imnet_by_citie.values(), ignore_index=True)

<class 'pandas.DataFrame'>
RangeIndex: 946 entries, 0 to 945
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   year                        946 non-null    int32  
 1   mean_temperature_c          946 non-null    float64
 2   max_temperature_c           946 non-null    float64
 3   min_temperature_c           946 non-null    float64
 4   total_rain_mm               946 non-null    float64
 5   sum_global_radiation_kj_m2  946 non-null    float64
 6   mean_relative_humidity_pct  946 non-null    float64
 7   max_relative_humidity_pct   946 non-null    float64
 8   min_relative_humidity_pct   946 non-null    float64
 9   code                        946 non-null    int64  
 10  city                        946 non-null    str    
dtypes: float64(8), int32(1), int64(1), str(1)
memory usage: 77.7 KB


In [45]:
%store df_inmet_final

Stored 'df_inmet_final' (DataFrame)
